# HWM Composite Sensitivity

**Purpose.** Exploratory sensitivity check for adding `hwm_heatwave_magnitude` alongside `hwa_heatwave_amplitude` in the Heat Risk bundle before any production methodology change lands.

**Scope.** District level, `historical / 1990-2010 / mean`. Baseline reconstruction is scored per state to match the persisted composite builder. The national-pooled view is intentionally secondary and caveated because production Heat Risk composites are normalized within each state.

**Read/write contract.** Reads existing processed masters only. Writes small exploratory outputs under `scratch/results/hwm_composite_sensitivity/`; it does not write to `IRT_DATA_DIR`.

**Important caveat.** `hwm` does not exist on disk yet. The three scenarios below proxy its normalized behavior, so results are a sensitivity band rather than a prediction.

## 1. Configuration

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd

from india_resilience_tool.analysis.bundle_scores import (
    BundleMetricSpec,
    compute_bundle_score_frame,
    normalized_metric_column,
)
from india_resilience_tool.compute.composite_metrics import (
    LEGACY_MASTER_FILENAMES,
    SUPPORTED_STAT,
    _build_wide_component_frame,
    _bundle_metric_specs,
    _discover_states_for_spec,
    _load_component_master,
    _required_id_columns,
)
from india_resilience_tool.config.composite_metrics import COMPOSITES_BY_BUNDLE
from india_resilience_tool.config.metrics_registry import METRICS_BY_SLUG
from india_resilience_tool.config.paths import get_paths_config, resolve_processed_root
from india_resilience_tool.data.master_loader import normalize_master_columns

BUNDLE = "Heat Risk"
LEVEL = "district"
REQUESTED_SCENARIO = "historical"
REQUESTED_PERIOD = "1990-2010"
FALLBACK_SCENARIO = "ssp585"
FALLBACK_PERIOD = "2040-2060"
STAT = "mean"
SEED = 20260715

HWA = "hwa_heatwave_amplitude"
HWM = "hwm_heatwave_magnitude"
TAS = "tas_annual_mean"
TXX = "txx_annual_max"
TN90P = "tn90p_warm_nights_pct"
EXTREMES = {TXX, TN90P, HWA}

paths = get_paths_config()
DATA_DIR = Path(os.getenv("IRT_DATA_DIR", paths.data_dir)).expanduser().resolve()
OUT_DIR = Path("scratch/results/hwm_composite_sensitivity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

spec = COMPOSITES_BY_BUNDLE[BUNDLE]
id_columns = list(_required_id_columns(LEVEL))

def _score_column_for(scenario: str, period: str) -> str:
    return f"{spec.composite_slug}__{scenario}__{period}__{STAT}"


def _persisted_composite_columns() -> set[str]:
    root = resolve_processed_root(spec.composite_slug, data_dir=DATA_DIR, mode="portfolio")
    columns: set[str] = set()
    for path in root.glob(f"*/{LEGACY_MASTER_FILENAMES[LEVEL]}"):
        parquet = path.with_suffix(".parquet")
        if parquet.exists():
            frame = pd.read_parquet(parquet)
            columns.update(str(c) for c in frame.columns if str(c).startswith(f"{spec.composite_slug}__"))
    return columns


persisted_score_columns = _persisted_composite_columns()
requested_score_col = _score_column_for(REQUESTED_SCENARIO, REQUESTED_PERIOD)
fallback_score_col = _score_column_for(FALLBACK_SCENARIO, FALLBACK_PERIOD)
if requested_score_col in persisted_score_columns:
    SCENARIO = REQUESTED_SCENARIO
    PERIOD = REQUESTED_PERIOD
elif fallback_score_col in persisted_score_columns:
    SCENARIO = FALLBACK_SCENARIO
    PERIOD = FALLBACK_PERIOD
else:
    if not persisted_score_columns:
        raise FileNotFoundError("No persisted Heat Risk composite score columns found for fidelity check.")
    first = sorted(persisted_score_columns)[0]
    _, SCENARIO, PERIOD, _ = first.split("__", 3)
score_col = _score_column_for(SCENARIO, PERIOD)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR : {OUT_DIR.resolve()}")
print(f"Composite: {spec.composite_slug}")
print(f"Requested slice: {REQUESTED_SCENARIO} / {REQUESTED_PERIOD} / {STAT}")
print(f"Effective slice : {SCENARIO} / {PERIOD} / {STAT}")
if (SCENARIO, PERIOD) != (REQUESTED_SCENARIO, REQUESTED_PERIOD):
    print("NOTE: Requested historical slice is not present in persisted composite parquet; using a persisted production slice for the fidelity-backed run.")
print(f"Baseline score column: {score_col}")

## 2. Helpers

In [ ]:
def _component_frames_for_state(state_name: str) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for metric_slug in spec.component_metric_slugs:
        frame = _load_component_master(metric_slug, level=LEVEL, state_name=state_name, data_dir=DATA_DIR)
        if frame is None or frame.empty:
            return {}
        frames[metric_slug] = frame
    return frames


def _wide_for_state(state_name: str) -> pd.DataFrame:
    frames = _component_frames_for_state(state_name)
    if not frames:
        return pd.DataFrame(columns=id_columns + list(spec.component_metric_slugs))
    return _build_wide_component_frame(frames, level=LEVEL, scenario=SCENARIO, period=PERIOD)


def _score_wide(wide: pd.DataFrame, metric_specs: list[BundleMetricSpec]) -> pd.DataFrame:
    return compute_bundle_score_frame(wide, metric_specs=metric_specs, id_columns=id_columns)


def _load_persisted_composite(state_name: str) -> pd.DataFrame:
    root = resolve_processed_root(spec.composite_slug, data_dir=DATA_DIR, mode="portfolio")
    path = root / state_name / LEGACY_MASTER_FILENAMES[LEVEL]
    parquet = path.with_suffix(".parquet")
    if not parquet.exists():
        raise FileNotFoundError(f"Missing persisted composite parquet: {parquet}")
    return normalize_master_columns(pd.read_parquet(parquet))


def _modified_metric_specs() -> list[BundleMetricSpec]:
    modified: list[BundleMetricSpec] = []
    for base in _bundle_metric_specs(spec):
        if base.slug in EXTREMES:
            modified.append(replace(base, weight=0.25 / 4.0))
        else:
            modified.append(base)
    modified.append(
        BundleMetricSpec(
            slug=HWM,
            label="Heatwave Magnitude (proxied mean exceedance)",
            column=HWM,
            weight=0.25 / 4.0,
            higher_is_worse=True,
        )
    )
    return modified


def _inject_hwm_raw_from_norm(wide: pd.DataFrame, norm_values: pd.Series) -> pd.DataFrame:
    # compute_bundle_score_frame will min-max normalize this synthetic raw column.
    out = wide.copy()
    out[HWM] = pd.to_numeric(norm_values, errors="coerce")
    return out


def _scenario_hwm_norm(wide: pd.DataFrame, baseline_score: pd.DataFrame, scenario_name: str, rng: np.random.Generator) -> pd.Series:
    if scenario_name == "S1_correlated_hwa":
        return baseline_score[normalized_metric_column(HWA)]
    if scenario_name == "S2_cold_favoring_tas_inverse":
        tas_norm = baseline_score[normalized_metric_column(TAS)]
        return 100.0 - pd.to_numeric(tas_norm, errors="coerce")
    if scenario_name == "S3_shuffled_hwa":
        source = baseline_score[normalized_metric_column(HWA)].to_numpy(dtype=float, copy=True)
        finite_idx = np.flatnonzero(np.isfinite(source))
        shuffled = source.copy()
        shuffled[finite_idx] = rng.permutation(source[finite_idx])
        return pd.Series(shuffled, index=baseline_score.index)
    raise ValueError(f"Unknown scenario: {scenario_name}")


def _rank_deltas(frame: pd.DataFrame, score_column: str, modified_column: str, group_column: str | None) -> pd.DataFrame:
    out = frame.copy()
    if group_column:
        out["baseline_rank"] = out.groupby(group_column)[score_column].rank(ascending=False, method="min")
        out["modified_rank"] = out.groupby(group_column)[modified_column].rank(ascending=False, method="min")
    else:
        out["baseline_rank"] = out[score_column].rank(ascending=False, method="min")
        out["modified_rank"] = out[modified_column].rank(ascending=False, method="min")
    out["rank_shift"] = out["modified_rank"] - out["baseline_rank"]
    out["abs_rank_shift"] = out["rank_shift"].abs()
    return out


def _corr(a: pd.Series, b: pd.Series, method: str) -> float:
    valid = pd.concat([a, b], axis=1).dropna()
    if len(valid) < 3:
        return float("nan")
    return float(valid.iloc[:, 0].corr(valid.iloc[:, 1], method=method))


def _distribution_summary(values: pd.Series) -> dict[str, float]:
    x = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if x.empty:
        return {"n": 0, "mean": np.nan, "std": np.nan, "p05": np.nan, "median": np.nan, "p95": np.nan, "min": np.nan, "max": np.nan}
    return {
        "n": int(x.size),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if x.size > 1 else 0.0,
        "p05": float(x.quantile(0.05)),
        "median": float(x.median()),
        "p95": float(x.quantile(0.95)),
        "min": float(x.min()),
        "max": float(x.max()),
    }

## 3. Reconstruct Baseline And Run Fidelity Gate

In [ ]:
states = _discover_states_for_spec(spec, level=LEVEL, data_dir=DATA_DIR)
print(f"Discovered states: {len(states)}")

baseline_specs = _bundle_metric_specs(spec)
wide_by_state: dict[str, pd.DataFrame] = {}
baseline_parts: list[pd.DataFrame] = []
persisted_parts: list[pd.DataFrame] = []

for state_name in states:
    wide = _wide_for_state(state_name)
    if wide.empty:
        continue
    wide_by_state[state_name] = wide
    scored = _score_wide(wide, baseline_specs)
    baseline_parts.append(scored.assign(state_partition=state_name))

    persisted = _load_persisted_composite(state_name)
    persisted_parts.append(persisted[id_columns + [score_col]].assign(state_partition=state_name))

baseline_recon = pd.concat(baseline_parts, ignore_index=True)
persisted = pd.concat(persisted_parts, ignore_index=True)

fidelity = baseline_recon[id_columns + ["bundle_score"]].merge(
    persisted[id_columns + [score_col]], on=id_columns, how="outer", validate="one_to_one"
)
delta = pd.to_numeric(fidelity["bundle_score"], errors="coerce") - pd.to_numeric(fidelity[score_col], errors="coerce")
nan_aligned = fidelity["bundle_score"].isna().equals(fidelity[score_col].isna())
max_abs_diff = float(delta.abs().max(skipna=True)) if delta.notna().any() else 0.0

print(f"Baseline rows: {len(baseline_recon):,}")
print(f"States scored: {len(wide_by_state):,}")
print(f"NaN aligned: {nan_aligned}")
print(f"Max abs diff vs persisted parquet: {max_abs_diff:.12g}")

assert nan_aligned, "Baseline reconstruction NaNs do not align with persisted composite parquet. Check id-key join, directionality, renorm divisor, or stale on-disk composite."
assert max_abs_diff < 1e-6, "Baseline reconstruction failed fidelity gate. Check id-key join, directionality, renorm divisor, or stale on-disk composite parquet."
assert not np.isinf(baseline_recon.select_dtypes(include=[np.number])).any().any(), "Baseline reconstruction contains infinite values."

baseline_recon.head()

## 4. Per-State Sensitivity Scenarios

In [ ]:
scenario_names = [
    "S1_correlated_hwa",
    "S2_cold_favoring_tas_inverse",
    "S3_shuffled_hwa",
]
modified_specs = _modified_metric_specs()

scenario_frames: dict[str, pd.DataFrame] = {}
rng = np.random.default_rng(SEED)

for scenario_name in scenario_names:
    parts: list[pd.DataFrame] = []
    for state_name, wide in wide_by_state.items():
        baseline_state = baseline_recon.loc[baseline_recon["state_partition"] == state_name].reset_index(drop=True)
        wide_state = wide.reset_index(drop=True)
        hwm_norm = _scenario_hwm_norm(wide_state, baseline_state, scenario_name, rng)
        modified_wide = _inject_hwm_raw_from_norm(wide_state, hwm_norm)
        modified_score = _score_wide(modified_wide, modified_specs).rename(
            columns={"bundle_score": "modified_score", "available_metric_count": "modified_available_metric_count"}
        )
        base_cols = baseline_state[id_columns + ["bundle_score", "available_metric_count"]].rename(
            columns={"bundle_score": "baseline_score", "available_metric_count": "baseline_available_metric_count"}
        )
        joined = base_cols.merge(
            modified_score[id_columns + ["modified_score", "modified_available_metric_count"]],
            on=id_columns,
            how="inner",
            validate="one_to_one",
        )
        joined["scenario"] = scenario_name
        joined["state_partition"] = state_name
        joined["score_delta"] = joined["modified_score"] - joined["baseline_score"]
        joined["available_metric_count_delta"] = joined["modified_available_metric_count"] - joined["baseline_available_metric_count"]
        parts.append(joined)
    frame = pd.concat(parts, ignore_index=True)
    frame = _rank_deltas(frame, "baseline_score", "modified_score", "state")
    scenario_frames[scenario_name] = frame

per_state_results = pd.concat(scenario_frames.values(), ignore_index=True)
assert not np.isinf(per_state_results.select_dtypes(include=[np.number])).any().any(), "Per-state results contain infinite values."

per_state_results.groupby("scenario")["score_delta"].agg(["count", "mean", "std", "min", "median", "max"])

## 5. Within-State Summary

In [ ]:
state_sizes = baseline_recon.groupby("state", dropna=False).size().rename("n_districts").reset_index()
small_states = state_sizes.loc[state_sizes["n_districts"] < 3, "state"].tolist()
print(f"States with <3 districts: {len(small_states)}")
if small_states:
    print(", ".join(map(str, small_states)))

rows: list[dict[str, object]] = []
state_corr_rows: list[dict[str, object]] = []
for scenario_name, frame in scenario_frames.items():
    dist = _distribution_summary(frame["score_delta"])
    rows.append({
        "view": "within_state",
        "scenario": scenario_name,
        **{f"delta_{k}": v for k, v in dist.items()},
        "median_abs_rank_shift": float(frame["abs_rank_shift"].median()),
        "p95_abs_rank_shift": float(frame["abs_rank_shift"].quantile(0.95)),
        "moved_gt_5": int((frame["abs_rank_shift"] > 5).sum()),
        "moved_gt_10": int((frame["abs_rank_shift"] > 10).sum()),
        "district_rows": int(len(frame)),
        "seed": SEED if scenario_name == "S3_shuffled_hwa" else np.nan,
    })
    for state_name, g in frame.groupby("state", dropna=False):
        if len(g) < 3:
            continue
        state_corr_rows.append({
            "scenario": scenario_name,
            "state": state_name,
            "n_districts": int(len(g)),
            "spearman": _corr(g["baseline_score"], g["modified_score"], "spearman"),
            "kendall": _corr(g["baseline_score"], g["modified_score"], "kendall"),
            "median_abs_rank_shift": float(g["abs_rank_shift"].median()),
            "p95_abs_rank_shift": float(g["abs_rank_shift"].quantile(0.95)),
        })

state_corr = pd.DataFrame(state_corr_rows)
corr_summary = state_corr.groupby("scenario").agg(
    states=("state", "count"),
    median_spearman=("spearman", "median"),
    median_kendall=("kendall", "median"),
    median_state_p95_abs_rank_shift=("p95_abs_rank_shift", "median"),
).reset_index()

summary = pd.DataFrame(rows).merge(corr_summary, on="scenario", how="left")
summary

## 6. Caveated National-Pooled Pass

In [ ]:
national_wide = pd.concat(wide_by_state.values(), ignore_index=True)
national_baseline = _score_wide(national_wide, baseline_specs).rename(
    columns={"bundle_score": "baseline_score", "available_metric_count": "baseline_available_metric_count"}
)

national_frames: dict[str, pd.DataFrame] = {}
national_rng = np.random.default_rng(SEED)
for scenario_name in scenario_names:
    hwm_norm = _scenario_hwm_norm(national_wide, national_baseline, scenario_name, national_rng)
    modified_wide = _inject_hwm_raw_from_norm(national_wide, hwm_norm)
    modified = _score_wide(modified_wide, modified_specs).rename(
        columns={"bundle_score": "modified_score", "available_metric_count": "modified_available_metric_count"}
    )
    frame = national_baseline[id_columns + ["baseline_score", "baseline_available_metric_count"]].merge(
        modified[id_columns + ["modified_score", "modified_available_metric_count"]],
        on=id_columns,
        how="inner",
        validate="one_to_one",
    )
    frame["scenario"] = scenario_name
    frame["score_delta"] = frame["modified_score"] - frame["baseline_score"]
    frame = _rank_deltas(frame, "baseline_score", "modified_score", None)
    national_frames[scenario_name] = frame

national_results = pd.concat(national_frames.values(), ignore_index=True)
national_rows = []
for scenario_name, frame in national_frames.items():
    dist = _distribution_summary(frame["score_delta"])
    national_rows.append({
        "view": "national_pooled_caveated",
        "scenario": scenario_name,
        **{f"delta_{k}": v for k, v in dist.items()},
        "spearman": _corr(frame["baseline_score"], frame["modified_score"], "spearman"),
        "kendall": _corr(frame["baseline_score"], frame["modified_score"], "kendall"),
        "median_abs_rank_shift": float(frame["abs_rank_shift"].median()),
        "p95_abs_rank_shift": float(frame["abs_rank_shift"].quantile(0.95)),
        "moved_gt_5": int((frame["abs_rank_shift"] > 5).sum()),
        "moved_gt_10": int((frame["abs_rank_shift"] > 10).sum()),
        "district_rows": int(len(frame)),
        "seed": SEED if scenario_name == "S3_shuffled_hwa" else np.nan,
    })
national_summary = pd.DataFrame(national_rows)
national_summary

## 7. Movers And Cold/Hot Contrast

In [ ]:
def _top_movers(frame: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    cols = ["state", "district", "baseline_score", "modified_score", "score_delta", "baseline_rank", "modified_rank", "rank_shift", "abs_rank_shift"]
    return frame.sort_values(["abs_rank_shift", "score_delta"], ascending=[False, False]).loc[:, cols].head(n)


for scenario_name in scenario_names:
    print(f"\n=== {scenario_name}: within-state movers ===")
    display(_top_movers(scenario_frames[scenario_name]))
    print(f"\n=== {scenario_name}: national-pooled movers (caveated) ===")
    display(_top_movers(national_frames[scenario_name]))

contrast_names = ["Lahul", "Spiti", "Leh", "Mangan", "Kinnaur", "Tawang", "Jalgaon", "Anand", "Kheda"]
contrast_pattern = "|".join(contrast_names)
contrast = national_results.loc[
    national_results["district"].astype(str).str.contains(contrast_pattern, case=False, na=False),
    ["scenario", "state", "district", "baseline_score", "modified_score", "score_delta", "baseline_rank", "modified_rank", "rank_shift"],
].sort_values(["scenario", "district"])
contrast

## 8. Write Exploratory Artifacts

In [ ]:
combined_summary = pd.concat([summary, national_summary], ignore_index=True, sort=False)
summary_path = OUT_DIR / "hwm_composite_sensitivity_summary.csv"
within_path = OUT_DIR / "hwm_composite_sensitivity_within_state_rows.csv"
national_path = OUT_DIR / "hwm_composite_sensitivity_national_pooled_rows.csv"
md_path = OUT_DIR / "hwm_composite_sensitivity_summary.md"

combined_summary.to_csv(summary_path, index=False)
per_state_results.to_csv(within_path, index=False)
national_results.to_csv(national_path, index=False)

def _markdown_table(frame: pd.DataFrame) -> str:
    try:
        return frame.to_markdown(index=False)
    except ImportError:
        return "```text\n" + frame.to_string(index=False) + "\n```"


lines = [
    "# HWM Composite Sensitivity Summary",
    "",
    f"Data dir: `{DATA_DIR}`",
    f"Rows: `{len(baseline_recon):,}` district rows across `{len(wide_by_state)}` states",
    f"Fidelity max abs diff vs persisted parquet: `{max_abs_diff:.12g}`",
    f"NaN aligned: `{nan_aligned}`",
    f"Shuffle seed: `{SEED}`",
    f"States with <3 districts: `{len(small_states)}`",
    "",
    "## Within-State View (Production-Matching Normalization)",
    _markdown_table(summary),
    "",
    "## National-Pooled View (Caveated, Not Production)",
    _markdown_table(national_summary),
    "",
    "## Notes",
    "- `S1_correlated_hwa` is the reweight-only floor because proxied HWM follows normalized HWA.",
    "- `S2_cold_favoring_tas_inverse` is an assumption-heavy anomaly-lens proxy using `100 - norm(tas_annual_mean)` per comparison frame.",
    "- `S3_shuffled_hwa` is a reproducible noise ceiling using the recorded seed.",
    "- The national-pooled pass normalizes raw component columns once across India and is not how production persisted composites are built.",
]
md_path.write_text("\n".join(lines), encoding="utf-8")

manifest = {
    "data_dir": str(DATA_DIR),
    "summary_csv": str(summary_path),
    "within_state_csv": str(within_path),
    "national_pooled_csv": str(national_path),
    "summary_md": str(md_path),
    "scenario": SCENARIO,
    "period": PERIOD,
    "stat": STAT,
    "seed": SEED,
    "baseline_rows": int(len(baseline_recon)),
    "states_scored": int(len(wide_by_state)),
    "max_abs_diff": max_abs_diff,
    "nan_aligned": bool(nan_aligned),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Wrote {summary_path}")
print(f"Wrote {within_path}")
print(f"Wrote {national_path}")
print(f"Wrote {md_path}")
combined_summary